In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('spotify_user_behavior_realistic_50000_rows.csv')


In [3]:
df.head()        # first 5 rows



,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
0,1,France,25,2021-08-19,Premium Duo,Active,0,0,No,No,4,10.13,Bollywood,Radio,Concert Alerts,Tablet,7,8
1,2,Indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,5,11.63,Latin,Podcasts,Lyrics Translation,Mobile,7,6
2,3,Italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,3,9.50,Bollywood,Lyrics,Better AI Recommendations,Desktop,6,5
3,4,Italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,4,13.16,Electronic,Playlists,Social Listening,Smart Speaker,11,8
4,5,Australia,18,2020-05-29,Free,Active,0,0,No,No,4,12.70,Indie,Daily Mix,Lyrics Translation,Tablet,10,11


In [4]:
df.info()        # structure of data


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   user_id                         50000 non-null  int64  
 1   country                         50000 non-null  object 
 2   age                             50000 non-null  int64  
 3   signup_date                     50000 non-null  object 
 4   subscription_type               50000 non-null  object 
 5   subscription_status             50000 non-null  object 
 6   months_inactive                 50000 non-null  int64  
 7   inactive_3_months_flag          50000 non-null  int64  
 8   ad_interaction                  50000 non-null  object 
 9   ad_conversion_to_subscription   50000 non-null  object 
 10  music_suggestion_rating_1_to_5  50000 non-null  int64  
 11  avg_listening_hours_per_week    50000 non-null  float64
 12  favorite_genre                  

In [5]:
df.describe()    # statistical summary


,user_id,age,months_inactive,inactive_3_months_flag,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,playlists_created,avg_skips_per_day
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,38.010280,1.533020,0.222460,3.644100,9.988986,8.002680,10.025920
std,14433.901067,12.984989,1.952082,0.415903,1.114424,3.968927,2.831571,3.165579
min,1.000000,16.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,12500.750000,27.000000,0.000000,0.000000,3.000000,7.280000,6.000000,8.000000
50%,25000.500000,38.000000,1.000000,0.000000,4.000000,9.980000,8.000000,10.000000
75%,37500.250000,49.000000,2.000000,0.000000,5.000000,12.680000,10.000000,12.000000
max,50000.000000,60.000000,18.000000,1.000000,5.000000,26.250000,23.000000,25.000000


In [6]:
df.shape         # rows, columns

(50000, 18)

In [7]:
df.isnull().sum()


,0
user_id,0
country,0
age,0
signup_date,0
subscription_type,0
subscription_status,0
months_inactive,0
inactive_3_months_flag,0
ad_interaction,0
ad_conversion_to_subscription,0


there are no missing values in any of our columns so fillna will not take that.

In [12]:
df = df.drop_duplicates()


In [13]:
df.duplicated().sum()


np.int64(0)

In [14]:
df.dtypes


,0
user_id,int64
country,object
age,int64
signup_date,object
subscription_type,object
subscription_status,object
months_inactive,int64
inactive_3_months_flag,int64
ad_interaction,object
ad_conversion_to_subscription,object


In [31]:
# Convert to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# Drop invalid dates
df = df.dropna(subset=['signup_date'])

# Create user age (days)
today = pd.Timestamp.today()
df['user_age_days'] = (today - df['signup_date']).dt.days


Churn Definition


In [32]:
df['is_churned'] = (df['subscription_status'] == 'Inactive').astype(int)


IQR Outlier Detection (Listening Hours)

In [33]:
Q1 = df['avg_listening_hours_per_week'].quantile(0.25)
Q3 = df['avg_listening_hours_per_week'].quantile(0.75)
IQR = Q3 - Q1


In [34]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


Flag outliers

In [35]:
df['is_outlier'] = (
    (df['avg_listening_hours_per_week'] < lower_bound) |
    (df['avg_listening_hours_per_week'] > upper_bound)
)


Remove outliers

In [36]:
df = df[
    (df['avg_listening_hours_per_week'] >= lower_bound) &
    (df['avg_listening_hours_per_week'] <= upper_bound)
]


Age Segmentation

In [37]:
bins = [0, 18, 24, 34, 44, 54, 100]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55+']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)


Engagement Segmentation

In [39]:
conditions = [
    df['avg_listening_hours_per_week'] < 10,
    (df['avg_listening_hours_per_week'] >= 10) & (df['avg_listening_hours_per_week'] < 30),
    df['avg_listening_hours_per_week'] >= 30
]

choices = ['Low', 'Medium', 'High']

df['engagement_level'] = np.select(conditions, choices, default='Undefined')

Active–Inactive Paradox Fix

In [40]:
df['likely_churned'] = np.where(
    (df['subscription_status'] == 'Active') &
    (df['months_inactive'] > 6),
    1,
    0
)


Adjust status

In [41]:
df['adjusted_status'] = df['subscription_status']

df.loc[df['likely_churned'] == 1, 'adjusted_status'] = 'Likely Churned'


Flag Synchronization

In [42]:
df['inactive_3_months_flag_corrected'] = (df['months_inactive'] >= 3).astype(int)

# Check mismatch
mismatch = df[df['inactive_3_months_flag'] != df['inactive_3_months_flag_corrected']]
print("Mismatch rows:", mismatch.shape[0])


Mismatch rows: 0


String Cleaning

In [43]:
cols = ['country', 'favorite_genre', 'primary_device']

for col in cols:
    df[col] = df[col].astype(str).str.strip().str.lower()


Tenure Bucket

In [45]:
df['tenure_months'] = df['user_age_days'] / 30

conditions = [
    df['tenure_months'] < 6,
    (df['tenure_months'] >= 6) & (df['tenure_months'] < 24),
    df['tenure_months'] >= 24
]

choices = ['New', 'Established', 'Loyal']

df['tenure_bucket'] = np.select(conditions, choices, default='Undefined')

Engagement Score
Listening hours = 50% importance
Playlists created = 50% importance

In [46]:
df['engagement_score'] = (
    df['playlists_created'] * 0.5 +
    df['avg_listening_hours_per_week'] * 0.5
)


Normalize score (optional but recommended)

In [47]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['engagement_score'] = scaler.fit_transform(df[['engagement_score']])


Final Validation

In [49]:
df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 49827 entries, 0 to 49999
Data columns (total 29 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   user_id                           49827 non-null  int64         
 1   country                           49827 non-null  object        
 2   age                               49827 non-null  int64         
 3   signup_date                       49827 non-null  datetime64[ns]
 4   subscription_type                 49827 non-null  object        
 5   subscription_status               49827 non-null  object        
 6   months_inactive                   49827 non-null  int64         
 7   inactive_3_months_flag            49827 non-null  int64         
 8   ad_interaction                    49827 non-null  object        
 9   ad_conversion_to_subscription     49827 non-null  object        
 10  music_suggestion_rating_1_to_5    49827 non-null  i

In [50]:
df.describe()

df[['user_age_days', 'age_group', 'engagement_level',
    'tenure_bucket', 'engagement_score',
    'adjusted_status']].head()

,user_age_days,age_group,engagement_level,tenure_bucket,engagement_score,adjusted_status
0,1705,25-34,Medium,Loyal,0.451694,Active
1,1414,18-24,Medium,Loyal,0.493699,Active
2,837,45-54,Low,Loyal,0.406049,Active
3,2794,45-54,Medium,Loyal,0.648558,Active
4,2152,<18,Medium,Loyal,0.607673,Active


In [52]:
df.to_csv('final_cleaned_spotify_data.csv', index=False)
